# Filter low quality paraphrases
Find out how many paraphrases are of low quality and filter them out.

Load paraphrases.

In [1]:
import os, sys
from nltk.stem.snowball import SnowballStemmer
import numpy as np
import pandas as pd
from pathlib import Path

# sys.path.append(os.path.abspath(".."))
from genai_detection.detectors.impostor import ImpostorDetector
from genai_detection.visualization.datasets import *
from genai_detection.config import CONFIG
from functools import partial
import gc
import datetime
import json
import textwrap
import pyreadstat
import collections
from datasets import load_from_disk

[nltk_data] Downloading package punkt_tab to /Users/klara/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
# student essays test
path2json1 = (
    Path(os.getcwd()).resolve().parent
    / CONFIG.SAVE_PATH
    / "impostor_scores/experiments/impostor_generator_comparison/complete_student_essays_test_dataset_1.json"
)

# student essays test for sim syn
path2json2 = (
    Path(os.getcwd()).resolve().parent
    / CONFIG.SAVE_PATH
    / "impostor_scores/experiments/impostor_generator_comparison/student_essays_naive_llm_test_impostor_scores_syn_sim.json"
)

# detection scenarios Student essays
path2json3 = (
    Path(os.getcwd()).resolve().parent
    / CONFIG.SAVE_PATH
    / "impostor_scores/experiments/detection_scenarios/student_essays_naive_llm_test_impostor_scores_detection_scenarios.json"
)

store_length = pd.DataFrame(
    columns=[
        "paraphraser",
        "prompt",
        "reference_text_length",
        "paraphrase_length",
        "smaller700",
        "reference_pair_id",
        "pair_id",
        "realtive_len_perc",
        "post_process_len_perc(qwen)",
    ]
)
reference_pair_ids = pd.DataFrame(
    columns=["reference_pair_id", "pair_id", "reference_text"]
)
store_garbage_texts = pd.DataFrame(
    columns=["reference_pair_id", "pair_id", "paraphraser", "garbage"]
)
garbage_texts = ["I sorry, but I can't help with that."]

extremest_len_paraphrases = pd.DataFrame(
    columns=[
        "reference_text_length",
        "paraphrase_length",
        "smaller700",
        "paraphraser",
        "reference_pair_id",
        "pair_id",
        "realtive_len_perc",
        "prompt",
        "paraphrase",
    ]
)

perfect_paraphrases = pd.DataFrame(
    columns=[
        "reference_text_length",
        "paraphrase_length",
        "smaller700",
        "paraphraser",
        "reference_pair_id",
        "pair_id",
        "realtive_len_perc",
        "prompt",
        "paraphrase",
    ]
)


# read json file
for path2json in [path2json1, path2json2, path2json3]:
    print("Reading json file:", path2json)
    with open(path2json, "r", encoding="utf-8") as f:
        df = json.load(f)

    for i in range(len(df)):  # uppermost list level: dictionary
        item = df[i]
        if "artificial_generation" in item and item["artificial_generation"]:
            # print("Skipping item", i, "because 'artificial_generation' is True, i.e. pair was generated.")
            continue
        if (
            "impostor_dict_naive_llm" not in item
            or item["impostor_dict_naive_llm"] is None
        ):
            # print("Skipping item", i, "because no 'impostor_dict_naive_llm' key found.")
            continue
        pair_dict = item["impostor_dict_naive_llm"]["text_pair_0"]  # dict with two keys

        for (
            no_in_pair,
            imp_dict,
        ) in pair_dict.items():  # dict containing reference text and paraphrases
            reference_text = imp_dict["reference_text"]
            paraphrases_dict = imp_dict["paraphrases"]
            # calculate word counts (split on whitespace)
            ref_len = len(reference_text.split())

            # add reference text info (ensuring no duplicates)
            reference_pair_ids = pd.concat(
                [
                    reference_pair_ids,
                    pd.DataFrame(
                        [
                            {
                                "reference_pair_id": i,  # position in the outermost list
                                "pair_id": no_in_pair,  # "0" or "1"
                                "reference_text": reference_text,
                            }
                        ]
                    ),
                ],
                ignore_index=True,
            )

            for impostor_name, para in paraphrases_dict.items():  # impostors
                split_name = impostor_name.split("_")
                paraphraser = split_name[-1]
                prompt = split_name[2]

                para_len = len(para.split())
                # check for garbage
                garbage = any(g in para for g in garbage_texts)

                relative_len_perc = np.round((para_len / ref_len) * 100, 2)

                processed_text_len_perc = relative_len_perc
                if "</think>" in para:
                    temp = para.split("</think>")[-1].strip()
                    processed_text_len_perc = np.round(
                        (len(temp.split()) / ref_len) * 100, 2
                    )

                # add length info
                store_length = pd.concat(
                    [
                        store_length,
                        pd.DataFrame(
                            [
                                {
                                    "reference_text_length": ref_len,
                                    "paraphrase_length": para_len,
                                    "smaller700": para_len < 700,
                                    "paraphraser": paraphraser,
                                    "prompt": prompt,
                                    "reference_pair_id": i,  # position in the outermost list
                                    "pair_id": no_in_pair,  # "0" or "1"
                                    "realtive_len_perc": relative_len_perc,
                                    "post_process_len_perc(qwen)": processed_text_len_perc,
                                }
                            ]
                        ),
                    ],
                    ignore_index=True,
                )
                if relative_len_perc > 110 or relative_len_perc < 10:
                    extremest_len_paraphrases = pd.concat(
                        [
                            extremest_len_paraphrases,
                            pd.DataFrame(
                                [
                                    {
                                        "reference_text_length": ref_len,
                                        "paraphrase_length": para_len,
                                        "smaller700": para_len < 700,
                                        "paraphraser": paraphraser,
                                        "prompt": prompt,
                                        "reference_pair_id": i,  # position in the outermost list
                                        "pair_id": no_in_pair,  # "0" or "1"
                                        "realtive_len_perc": relative_len_perc,
                                        "paraphrase": para,
                                    }
                                ]
                            ),
                        ],
                        ignore_index=True,
                    )
                elif relative_len_perc > 95 and relative_len_perc < 105:
                    perfect_paraphrases = pd.concat(
                        [
                            perfect_paraphrases,
                            pd.DataFrame(
                                [
                                    {
                                        "paraphraser": paraphraser,
                                        "prompt": prompt,
                                        "reference_text_length": ref_len,
                                        "paraphrase_length": para_len,
                                        "smaller700": para_len < 700,
                                        "reference_pair_id": i,  # position in the outermost list
                                        "pair_id": no_in_pair,  # "0" or "1"
                                        "realtive_len_perc": relative_len_perc,
                                        "paraphrase": para,
                                    }
                                ]
                            ),
                        ],
                        ignore_index=True,
                    )

                # add garbage info
                store_garbage_texts = pd.concat(
                    [
                        store_garbage_texts,
                        pd.DataFrame(
                            [
                                {
                                    "paraphraser": paraphraser,
                                    "garbage": garbage,
                                    "reference_pair_id": i,  # position in the outermost list
                                    "pair_id": no_in_pair,  # "0" or "1"
                                }
                            ]
                        ),
                    ],
                    ignore_index=True,
                )


display(store_length)
# display(store_garbage_texts)
# display(reference_pair_ids)
print("Number of garbage texts:", store_garbage_texts["garbage"].sum())

Reading json file: /Users/klara/Developer/Uni/MA/artificial-authorship-verification/results/impostor_scores/experiments/impostor_generator_comparison/complete_student_essays_test_dataset_1.json


/var/folders/hk/dwphr0p14vn00wrnlw8bxjpw0000gn/T/ipykernel_45888/3051013312.py:136: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  store_length = pd.concat(
/var/folders/hk/dwphr0p14vn00wrnlw8bxjpw0000gn/T/ipykernel_45888/3051013312.py:158: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  extremest_len_paraphrases = pd.concat(
/var/folders/hk/dwphr0p14vn00wrnlw8bxjpw0000gn/T/ipykernel_45888/3051013312.py:180: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is depr

Reading json file: /Users/klara/Developer/Uni/MA/artificial-authorship-verification/results/impostor_scores/experiments/impostor_generator_comparison/student_essays_naive_llm_test_impostor_scores_syn_sim.json
Reading json file: /Users/klara/Developer/Uni/MA/artificial-authorship-verification/results/impostor_scores/experiments/detection_scenarios/student_essays_naive_llm_test_impostor_scores_detection_scenarios.json


,reference_text_length,paraphrase_length,smaller700,paraphraser,reference_pair_id,pair_id,realtive_len_perc,prompt,post_process_len_perc(qwen)
0,794,22,True,mistral-large-instruct,0,0,2.77,prompt0,2.77
1,794,242,True,mistral-large-instruct,0,0,30.48,prompt1,30.48
2,794,127,True,meta-llama-3.1-8b-instruct,0,0,15.99,prompt0,15.99
3,794,175,True,meta-llama-3.1-8b-instruct,0,0,22.04,prompt1,22.04
4,794,8,True,openai-gpt-oss-120b,0,0,1.01,prompt0,1.01
...,...,...,...,...,...,...,...,...,...
1046,935,548,True,meta-llama-3.1-8b-instruct,72,1,58.61,prompt1,58.61
1047,707,519,True,qwen3-32b,79,0,73.41,prompt0,37.34
1048,707,952,False,qwen3-32b,79,0,134.65,prompt1,58.27
1049,927,765,False,qwen3-32b,79,1,82.52,prompt0,30.74


Number of garbage texts: 0


In [3]:
import seaborn as sns
import matplotlib.pyplot as plt

for scale in ["linear", "log"]:
    plt.figure(figsize=(10, 6))
    sns.boxplot(
        x="paraphraser", y="mean_processed_qwen", data=store_length, hue="prompt"
    )

    if scale == "log":
        plt.yscale("log")  # log scale to expand lower region
    plt.title(
        f"Distribution of Relative Length Percentage per Paraphraser ({scale} scale)"
    )
    plt.xlabel("Paraphraser")
    plt.ylabel("Relative Length Percentage")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    plt.close()

ValueError: Could not interpret value `mean_processed_qwen` for `y`. An entry with this name does not appear in `data`.

<Figure size 1000x600 with 0 Axes>

In [ ]:
import numpy as np

gt_store_length = (
    store_length[
        ["paraphraser", "prompt", "realtive_len_perc", "post_process_len_perc(qwen)"]
    ]
    .groupby(["paraphraser", "prompt"])
    .agg(
        mean_relative_len_perc=("realtive_len_perc", "mean"),
        std_relative_len_perc=("realtive_len_perc", "std"),
        mean_processed_qwen=("post_process_len_perc(qwen)", "mean"),
        std_processed_qwen=("post_process_len_perc(qwen)", "std"),
        count=("realtive_len_perc", "size"),
    )
    .reset_index()
)

# round floats for readability
gt_store_length = gt_store_length.applymap(
    lambda x: np.round(x, 2) if isinstance(x, float) else x
)
gt_store_length

/var/folders/hk/dwphr0p14vn00wrnlw8bxjpw0000gn/T/ipykernel_45707/1944799240.py:17: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  gt_store_length = gt_store_length.applymap(


,paraphraser,prompt,mean_relative_len_perc,std_relative_len_perc,mean_processed_qwen,std_processed_qwen,count
0,meta-llama-3.1-8b-instruct,prompt0,39.93,52.64,39.93,52.64,135
1,meta-llama-3.1-8b-instruct,prompt1,40.27,24.21,40.27,24.21,124
2,mistral-large-instruct,prompt0,1.89,1.00,1.89,1.00,138
3,mistral-large-instruct,prompt1,13.09,17.96,13.09,17.96,129
4,openai-gpt-oss-120b,prompt0,5.53,13.47,5.53,13.47,139
5,openai-gpt-oss-120b,prompt1,19.21,25.00,19.21,25.00,129
6,qwen3-32b,prompt0,88.36,70.02,18.68,24.09,134
7,qwen3-32b,prompt1,95.73,47.72,38.34,15.64,123


Relative length is computed as $\frac{len(paraphrase)}{len(reference)}\times 100\%$.

```python
(para_len / ref_len) * 100
```
Hence, a relative length of 100% means that the paraphrase has the same length as the reference. 
A relative length of 50% means that the paraphrase is half as long as the reference, and a relative length of 200% means that the paraphrase is twice as long as the reference.

_Count_ denotes the number of samples considered for the statistics.

We computed mean and standard deviation of the relative lengths per paraphraser and prompt.
All statistics are computed on samples from the Student Essay dataset (where candidates and disputed texts are authored by humans).

In [ ]:
save_path = (
    Path(os.getcwd()).resolve().parent
    / CONFIG.SAVE_PATH
    / "post_hoc_paraphrase_analysis"
)
save_path.mkdir(parents=True, exist_ok=True)
gt_store_length.to_csv(
    save_path / "relative_length_perc_per_paraphraser_prompt.csv", index=False
)
print("Saved to", save_path / "relative_length_perc_per_paraphraser_prompt.csv")

Saved to /Users/klara/Developer/Uni/MA/artificial-authorship-verification/results/post_hoc_paraphrase_analysis/relative_length_perc_per_paraphraser_prompt.csv


Are the longest `qwen3-32b` paraphrases of low quality?

Yes, they contain the step-by-step process of replacing words with synonyms.

In [ ]:
qwen_extremes = extremest_len_paraphrases[
    extremest_len_paraphrases["paraphraser"] == "qwen3-32b"
]
print("Number of long qwen3-32b paraphrases:", len(qwen_extremes))
print(qwen_extremes.columns)

top_qwen = qwen_extremes.sort_values(by="paraphrase_length", ascending=False)[
    ["paraphrase", "paraphrase_length"]
].head(5)

for _, row in top_qwen.iterrows():
    p = row["paraphrase"]
    l = row["paraphrase_length"]

    # show first and last 100 characters
    print("Number of words:", l)
    print(
        textwrap.fill(p[100:], width=80), "...\n...", textwrap.fill(p[-100:], width=80)
    )
    print("-" * 80)

Number of long qwen3-32b paraphrases: 58
Index(['reference_text_length', 'paraphrase_length', 'smaller700',
       'paraphraser', 'reference_pair_id', 'pair_id', 'realtive_len_perc',
       'prompt', 'paraphrase'],
      dtype='object')
Number of words: 3950
ain subject, verb, and object, replacing them with synonyms, and adjusting
grammar naturally. The original text is a long, stream-of-consciousness
monologue with various topics.   First, I need to parse each sentence to find
the main elements. For example, the first sentence is "I really wish I was in
Florida right about now." The main subject is "I," the verb is "wish," and the
object is "to be in Florida." A synonym for "wish" could be "want," and "in
Florida" could be rephrased as "there in Florida." So, "I really want to be in
Florida right now."  Next, the user mentions missing a friend at Disney World.
The main subject is "I," the verb is "miss," and the object is "my friend."
Replacing "miss" with "long for" and rephrasing t

Are perfect paraphrases (in terms of length) of low quality? 
We use this as a heuristic for paraphrases of similar length as the reference.

4 out of 5 perfect paraphrases (in terms of length) generated by `qwen3-32b` include a `</think>`.
Before this sign all text is chain-of-thought explaining how to replace words with synonyms.
After this sign the actual paraphrase starts.

In [ ]:
qwen_perfects = perfect_paraphrases[perfect_paraphrases["paraphraser"] == "qwen3-32b"]
print("Number of long qwen3-32b paraphrases:", len(qwen_perfects))

top_qwen = qwen_perfects[["paraphrase", "paraphrase_length"]].head(5)

for _, row in top_qwen.iterrows():
    p = row["paraphrase"]
    l = row["paraphrase_length"]

    # show first and last 100 characters
    print("Number of words:", l)
    print(
        textwrap.fill(p[100:], width=80), "...\n...", textwrap.fill(p[-100:], width=80)
    )
    print("-" * 80)

Number of long qwen3-32b paraphrases: 29
Number of words: 725
ging its meaning, use different words and vary sentence structure, keep the tone
consistent, and maintain similar length. The original text is quite long and has
a lot of specific names and a somewhat informal, even vulgar tone.  First, I
need to understand the main points. Susan is irritated because Irene is taking
credit for her discovery at Warford Labs involving monkey sperm that boosts
muscle in humans. Susan has worked there for 8 years, did tests on horse jockeys
with monkey sperm, and has a history with Ringo Bonkers, a jockey who won a big
race. Susan and Irene have a rivalry, especially over Ringo. The story
culminates in Irene getting the glory in a magazine feature instead of Susan.
Now, I need to rephrase this. Start by replacing "irritated" with a synonym like
"annoyed" or "infuriated." "Credit for their new found glory" could be
"recognition for their newfound acclaim." "Monkey sperm that actually increases
mu

Try omitting `</think>` and the text before it from the paraphrase.

In [ ]:
import textwrap

# Post-process the paraphrase text
qwen_perfects["post_processed_text"] = qwen_perfects.apply(
    lambda row: (
        row["paraphrase"].split("</think>")[-1].strip()
        if "</think>" in row["paraphrase"]
        else row["paraphrase"]
    ),
    axis=1,
)

# Select top 5 by length (or just first 5 rows)
top_qwen = qwen_perfects[["post_processed_text", "paraphrase_length"]].head(5)

for _, row in top_qwen.iterrows():
    p = row["post_processed_text"]
    l = len(p.split())

    print(
        "Number of words before/ after processing:", row["paraphrase_length"], " / ", l
    )
    print("-" * 80)
    # wrap full text every 80 characters
    print(textwrap.fill(p, width=80))
    print("=" * 80)
    print("=" * 80)

Number of words before/ after processing: 725  /  258
--------------------------------------------------------------------------------
Susan glared at Irene, seething, aware that Irene would siphon the accolades for
their accidental breakthrough—monkey sperm that amplified human muscle mass—a
phenomenon both unprecedented and controversial among certain medical experts.
Susan craved the recognition, the prestige, the very substance she’d spent eight
years at Warford Labs perfecting through her studies on equine jockeys. She’d
dosed them with copious monkey sperm to boost training, leveraging the lab’s
ties to racetrack royalty like Ringo Bonkers and his steed Sebastian’s Carriage,
alongside Dr. Wilfy Ruggles and his champion Diaper Delivery. When Irene, the
backstabbing harlot, attempted to hijack Susan’s discovery, she knew she had to
crush her. Irene had long targeted Ringo, Susan’s ex-lover, who’d become
untouchable after his Frankfurt Derby Downs victory last summer. His horse, a
d

/var/folders/hk/dwphr0p14vn00wrnlw8bxjpw0000gn/T/ipykernel_45707/3659765756.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  qwen_perfects["post_processed_text"] = qwen_perfects.apply(
